# SER - Experiment 07: final test evaluation + per-corpus breakdown

Run this **after** notebook 06 finishes. It does three things:

1. Reads `sweep_results.json` and picks the winner **by validation accuracy**
   - never by test accuracy. That single rule is what keeps the number
   honest.
2. Evaluates the winner on the **test set, exactly once**, alongside the
   `base` control for a like-for-like comparison.
3. Produces a **per-corpus breakdown** for both, which explains where the
   combined figure comes from and gives a legitimate single-corpus headline.

No training happens here - notebook 06 already saved every candidate model, so
this takes about five minutes.

**Attach:** the four corpora, `ser-feature-cache`, **and notebook 06's
output** (which carries `sweep_results.json` and the `sweep_*.keras` models).

**Accelerator: GPU** (or CPU - it only does inference).

In [ ]:
import glob
import json
import os
import shutil
import sys

import numpy as np
import pandas as pd
import tensorflow as tf
import keras

print("TF", tf.__version__, "| Keras", keras.__version__)

In [ ]:
REPO = "https://github.com/Eldorado5002/ser.git"

if not os.path.exists("/kaggle/working/ser"):
    !git clone -q {REPO} /kaggle/working/ser

sys.path.insert(0, "/kaggle/working/ser")
os.chdir("/kaggle/working/ser")
!git log --oneline -1

In [ ]:
DATA_ROOT = "/kaggle/working/ser/data"

CANONICAL = {
    "RAVDESS": "audio_speech_actors_01-24",
    "TESS":    "TESS Toronto emotional speech set data",
    "SAVEE":   "ALL",
    "CREMA-D": "AudioWAV",
}


def find_canonical(target):
    hits = []
    for root, dirs, _ in os.walk("/kaggle/input"):
        for d in dirs:
            if d.lower() == target.lower():
                hits.append(os.path.join(root, d))
    return sorted(hits)[0] if hits else None


os.makedirs(DATA_ROOT, exist_ok=True)
for name, target in CANONICAL.items():
    src = find_canonical(target)
    assert src is not None, f"MISSING INPUT for {name}: no '{target}' found"
    dst = os.path.join(DATA_ROOT, name)
    if os.path.islink(dst):
        os.unlink(dst)
    elif os.path.exists(dst):
        shutil.rmtree(dst)
    os.symlink(src, dst)
    print(f"{name:9s} -> {src}")

In [ ]:
# ---------------------------------------------------------------------------
# Locate notebook 06's output and pick the winner BY VALIDATION ACCURACY.
# ---------------------------------------------------------------------------
sweep_json = None
for root, dirs, files in os.walk("/kaggle/input"):
    if "sweep_results.json" in files:
        sweep_json = os.path.join(root, "sweep_results.json")
        break

assert sweep_json, (
    "sweep_results.json not found. Attach notebook 06's output as an input "
    "dataset (Add Input -> Your Work -> the 06 notebook).")

results = json.load(open(sweep_json))
print(f"loaded {len(results)} candidates from {sweep_json}\n")

df = pd.DataFrame(results).sort_values("val_accuracy", ascending=False)
print(df[["tag", "val_accuracy", "train_accuracy", "gap", "params",
          "best_epoch"]].to_string(index=False))

WINNER = df.iloc[0]["tag"]
print(f"\nWINNER (by validation): {WINNER}")

MODEL_DIR = os.path.dirname(sweep_json)
models = {}
for root, dirs, files in os.walk("/kaggle/input"):
    for f in files:
        if f.startswith("sweep_") and f.endswith(".keras"):
            models[f[len("sweep_"):-len(".keras")]] = os.path.join(root, f)
print("saved models found:", sorted(models))

EVALUATE = [t for t in ("base", WINNER) if t in models]
assert EVALUATE, "no saved sweep models found in the attached input"
print("will evaluate:", EVALUATE)

In [ ]:
import config
from data_loader import build_metadata, split_metadata
from augmentation import plan_augmentation
from features import build_feature_matrix, df_to_items
from utils import StreamScalers, set_seed

config.CACHE_DIR = "/kaggle/working/features_cache"
os.makedirs(config.CACHE_DIR, exist_ok=True)
for root, dirs, files in os.walk("/kaggle/input"):
    for f in files:
        if f.endswith(".npz"):
            shutil.copy(os.path.join(root, f), config.CACHE_DIR)

meta = build_metadata(strict=True)
assert len(meta) == 12162
train_df, val_df, test_df = split_metadata(meta)
set_seed(config.RANDOM_SEED)

# Scalers must be refitted exactly as notebook 06 did: same cache, same
# deterministic plan, therefore identical statistics.
def train_features(kind):
    if kind == "3x":
        orig = config.TARGET_TRAIN_SIZE
        config.TARGET_TRAIN_SIZE = 3 * len(train_df)
        items = plan_augmentation(train_df, emotion_aware=False)
        config.TARGET_TRAIN_SIZE = orig
        return build_feature_matrix(items, desc="train_uniform3x")
    items = plan_augmentation(train_df, emotion_aware=False)
    return build_feature_matrix(items, desc="train_uniform")


test_feats = build_feature_matrix(df_to_items(test_df), desc="test")
print(f"\ntest set: {test_feats['mfcc'].shape[0]:,} samples "
      f"- being used for the FIRST and ONLY time")

In [ ]:
from evaluate import evaluate_predictions

by_tag = {r["tag"]: r for r in results}
outputs = {}

for tag in EVALUATE:
    kind = by_tag[tag]["data"]
    sc = StreamScalers().fit(train_features(kind))
    x_test = sc.transform(test_feats)

    model = tf.keras.models.load_model(models[tag], compile=False)
    y_prob = model.predict(x_test, batch_size=config.BATCH_SIZE, verbose=0)

    out_dir = f"/kaggle/working/final/{tag}"
    os.makedirs(out_dir, exist_ok=True)
    print(f"\n{'=' * 70}\n[test] {tag}\n{'=' * 70}")
    m = evaluate_predictions(test_feats["y"], y_prob, out_dir=out_dir,
                             prefix="test")
    outputs[tag] = {"metrics": m, "y_prob": y_prob,
                    "params": int(model.count_params())}

In [ ]:
# ---------------------------------------------------------------------------
# Per-corpus breakdown. df_to_items preserves row order and
# build_feature_matrix preserves it too, so test_df aligns with the
# predictions positionally.
# ---------------------------------------------------------------------------
corpora = test_df["corpus"].values
y_true = test_feats["y"]
assert len(corpora) == len(y_true), "row alignment broken"

print("\n" + "=" * 78)
print("  PER-CORPUS ACCURACY (test set)")
print("=" * 78)
header = f"  {'corpus':10s} {'n':>6s}"
for tag in EVALUATE:
    header += f" {tag:>16s}"
print(header)
print("  " + "-" * 74)

percorpus = {}
for c in ["RAVDESS", "TESS", "SAVEE", "CREMA-D"]:
    mask = corpora == c
    if mask.sum() == 0:
        continue
    line = f"  {c:10s} {int(mask.sum()):6d}"
    percorpus[c] = {"n": int(mask.sum())}
    for tag in EVALUATE:
        pred = np.argmax(outputs[tag]["y_prob"][mask], axis=1)
        acc = float((pred == y_true[mask]).mean())
        percorpus[c][tag] = acc
        line += f" {acc*100:15.2f}%"
    print(line)

print("  " + "-" * 74)
line = f"  {'COMBINED':10s} {len(y_true):6d}"
for tag in EVALUATE:
    line += f" {outputs[tag]['metrics']['accuracy']*100:15.2f}%"
print(line)
print("=" * 78)

In [ ]:
BASE_REPORTED = 0.579531   # notebook 02, the reproduction in the report

print("\n" + "=" * 78)
print("  FINAL RESULT")
print("=" * 78)
for tag in EVALUATE:
    m = outputs[tag]["metrics"]
    lo, hi = m["accuracy_95ci"]
    print(f"  {tag:20s} acc {m['accuracy']*100:6.2f}%  "
          f"[{lo*100:.2f}, {hi*100:.2f}]  macroF1 {m['macro_f1']:.4f}  "
          f"MCC {m['mcc']:.4f}  {outputs[tag]['params']:,} params")

if WINNER in outputs and "base" in outputs:
    d = outputs[WINNER]["metrics"]["accuracy"] - outputs["base"]["metrics"]["accuracy"]
    print(f"\n  improvement over base: {d*100:+.2f} points")
print(f"  report's reproduction figure: {BASE_REPORTED*100:.2f}%")
print("=" * 78)

payload = {
    "winner": WINNER,
    "selected_by": "validation accuracy (test set never used for selection)",
    "sweep": results,
    "test": {t: {k: v for k, v in outputs[t]["metrics"].items()
                 if k != "y_prob"} for t in EVALUATE},
    "params": {t: outputs[t]["params"] for t in EVALUATE},
    "per_corpus": percorpus,
}
with open("/kaggle/working/final_results.json", "w") as f:
    json.dump(payload, f, indent=2, default=float)
print("\nwrote /kaggle/working/final_results.json")

for name in CANONICAL:
    link = os.path.join(DATA_ROOT, name)
    if os.path.islink(link):
        os.unlink(link)
print("output:", sorted(os.listdir("/kaggle/working")))